# AI Ticket Auto Assignment System

This notebook demonstrates the process of training a machine learning model to automatically classify support tickets into their respective categories.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

## 1. Load Dataset

In [2]:
df = pd.read_csv('../dataset/tickets.csv')
df.head()

## 2. Basic EDA and Data Cleaning

In [3]:
print("Missing values:\n", df.isnull().sum())
df = df.dropna()

print("\nCategory distribution:")
print(df['category'].value_counts())

In [4]:
plt.figure(figsize=(10,6))
sns.countplot(x='category', data=df)
plt.title('Distribution of Ticket Categories')
plt.show()

## 3. Train-Test Split

In [5]:
X = df['ticket_description']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

## 4. Feature Extraction (TF-IDF)
We use TF-IDF (Term Frequency-Inverse Document Frequency) to convert text into numerical features.

In [6]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## 5. Train Logistic Regression Model

In [7]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)

## 6. Model Evaluation

In [8]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [9]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 7. Test Model with Custom Inputs

In [10]:
test_tickets = [
    "My internet is not working even though my Wi-Fi is connected.",
    "I was charged twice for my subscription.",
    "I forgot my password."
]

test_tfidf = vectorizer.transform(test_tickets)
predictions = model.predict(test_tfidf)
probs = model.predict_proba(test_tfidf)

for ticket, pred, prob in zip(test_tickets, predictions, probs):
    confidence = max(prob) * 100
    print(f"Ticket: {ticket}")
    print(f"Prediction: {pred} (Confidence: {confidence:.2f}%)\n")

## 8. Save Model and Vectorizer

In [11]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/ticket_classifier.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print("Model and Vectorizer saved successfully!")